# T15 — Baseline evaluacija

Pokreće originalni `Qwen/Qwen2.5-1.5B-Instruct` prije fine-tuninga na fiksnom T14 skupu od 60 pitanja. RAG se ne koristi. Rezultati se upisuju u `artifacts/results/baseline_results.jsonl` i skripta može nastaviti nakon prekida.

## 1. Colab GPU provjera

U Colabu prvo uključiti **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi

In [ ]:
import torch
assert torch.cuda.is_available(), "CUDA GPU nije dostupan."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Instalacija projekta i zavisnosti

Notebook treba pokrenuti iz korijena kloniranog repozitorijuma.

In [ ]:
!pip install -q -r requirements-lock.txt
!pip install -q -e .

## 3. Provjera T14 inputa

Mora postojati tačno 60 fiksnih evaluacionih scenarija.

In [ ]:
import json
from pathlib import Path

questions_path = Path("data/evaluation/evaluation_questions.jsonl")
questions = [
    json.loads(line)
    for line in questions_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
assert len(questions) == 60
assert len({q["question_id"] for q in questions}) == 60
print("T14 input: OK — 60 scenarija")

## 4. Baseline evaluacija

Konfiguracija iz T15: `seed=42`, `temperature=0.2`, `top_p=0.9`, `max_new_tokens=300`. Model se učitava u istoj 4-bitnoj NF4 konfiguraciji kao T04. Ako se Colab prekine, samo ponovo pokrenuti ovu ćeliju; postojeći `question_id` biće preskočeni.

In [ ]:
!python -m bih_guide.evaluation.run_evaluation \
  --questions data/evaluation/evaluation_questions.jsonl \
  --output artifacts/results/baseline_results.jsonl

## 5. Završna provjera rezultata

T15 je spreman za commit tek kada postoji 60 validnih baseline rezultata.

In [ ]:
results_path = Path("artifacts/results/baseline_results.jsonl")
results = [
    json.loads(line)
    for line in results_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

required = {
    "question_id", "prompt", "response", "model_name",
    "seed", "generation_config", "conversation"
}

assert len(results) == 60
assert len({r["question_id"] for r in results}) == 60
assert all(required <= set(r) for r in results)
assert all(str(r["response"]).strip() for r in results)

print("Baseline rezultati: OK — 60/60")
print("Model:", results[0]["model_name"])
print("Seed:", results[0]["seed"])
print("Generation config:", results[0]["generation_config"])

In [ ]:
# Sačuvaj ovaj output u notebooku prije T15 commita.
!git status --short
!wc -l artifacts/results/baseline_results.jsonl